In [1]:
import pandas as pd

df = pd.read_json("../../datas/vps/detected.json", lines=True)
# df = pd.read_json("../detected.json", lines=True)

df.head(5)

,method,path,headers,uuid,peer,status,cookies,response_msg,timestamp,post_data
0,GET,/,"{'host': 'localhost', 'user-agent': 'curl/7.68...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '172.18.0.1', 'port': 53212}",200,{'sess_uuid': None},"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:30:36.539037,NaN
1,GET,/,"{'host': '202.10.35.215', 'connection': 'keep-...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '180.248.32.210', 'port': 30702}",200,{'sess_uuid': '7ee68b0c-9f55-47e5-b872-078d30b...,"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:31:01.702421,NaN
2,GET,/stylesheets/jquery/jquery-ui-1.11.0.css?15286...,"{'host': '202.10.35.215', 'connection': 'keep-...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '180.248.32.210', 'port': 30702}",200,{'sess_uuid': 'c3f38f1d-a1be-477a-8692-9369ce7...,"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:31:01.860896,NaN
3,GET,/stylesheets/application.css?1528612569,"{'host': '202.10.35.215', 'connection': 'keep-...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '180.248.32.210', 'port': 30709}",200,{'sess_uuid': 'c3f38f1d-a1be-477a-8692-9369ce7...,"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:31:02.008505,NaN
4,GET,/stylesheets/responsive.css?1528612569,"{'host': '202.10.35.215', 'connection': 'keep-...",64644ea7-b358-4208-9b28-e5aaad5e00de,"{'ip': '180.248.32.210', 'port': 8026}",200,{'sess_uuid': 'c3f38f1d-a1be-477a-8692-9369ce7...,"{'version': '0.6.0', 'response': {'message': {...",2025-06-14 07:31:02.020142,NaN


In [2]:
attacks_matrix = []
tmp = []
sess_uuids_attacks = {
  # "{sess_uuid}": []
}
sess_uuids = []

for _, row in df.iterrows():
  sess_uuid = row.get("cookies", {}).get("sess_uuid") or row.get("response_msg", {}).get("response", {}).get("message", {}).get("sess_uuid")
  detection = row.get("response_msg", {}).get("response", {}).get("message", {}).get("detection", {})
  attack_type = detection.get("name", "")

  if sess_uuid in sess_uuids:
    sess_uuids_attacks[sess_uuid].append(attack_type)
  else:
    sess_uuids_attacks[sess_uuid] = [attack_type]
    sess_uuids.append(sess_uuid)

attacks_matrix = list(sess_uuids_attacks.values())
# for paths in sess_uuids_attacks.values():
#   attacks_matrix.append(paths)

sess_uuids_attacks
# attacks_matrix
# print(len(attacks_matrix), len(sess_uuids_paths.keys()))

{'806c8d58-b04f-4634-b107-34877af9d1d0': ['index'],
 '7ee68b0c-9f55-47e5-b872-078d30b9116b': ['index'],
 'c3f38f1d-a1be-477a-8692-9369ce7d2c44': ['index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index'],
 'f278a7c1-0785-48d7-bf05-539b1f48cc0c': ['index'],
 '35356d10-0d1c-47ab-b018-36f62dd2a08f': ['lfi'],
 '1e975ea3-2f2c-4183-a4ca-7637ddbe1eec': ['index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index'],
 '7c28e45f-7aaa-42fa-b44d-0216a25150ab': ['index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index'],
 '47685c7c-17cb-4303-8a1e-5e99a9e1f9bd': ['xss',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index'],
 '09e54ba6-8322-4a86-afa8-a01dc89cb59b': ['lfi',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index'],
 '98ea628e-45bf-4a62-a0ee-3fab57dbf91d': ['index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index',
  'index'],
 '

In [3]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth

te = TransactionEncoder()
te_ary = te.fit(attacks_matrix).transform(attacks_matrix)
df = pd.DataFrame(te_ary, columns=te.columns_)

df.head(10)

,cmd_exec,index,lfi,sqli,unknown,xss
0,False,True,False,False,False,False
1,False,True,False,False,False,False
2,False,True,False,False,False,False
3,False,True,False,False,False,False
4,False,False,True,False,False,False
5,False,True,False,False,False,False
6,False,True,False,False,False,False
7,False,True,False,False,False,True
8,False,True,True,False,False,False
9,False,True,False,False,False,False


In [4]:
# frequent_itemsets = fpgrowth(df, min_support=0.3, use_colnames=True)
frequent_itemsets = fpgrowth(df, min_support=0.00001, use_colnames=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(lambda x: len(x))
### alternatively:
#frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
#frequent_itemsets = fpmax(df, min_support=0.6, use_colnames=True)

frequent_itemsets

,support,itemsets,length
0,0.950980,(index),1
1,0.019608,(lfi),1
2,0.009804,(xss),1
3,0.009804,(sqli),1
4,0.039216,(unknown),1
5,0.009804,(cmd_exec),1
6,0.009804,"(index, lfi)",2
7,0.009804,"(index, xss)",2
8,0.019608,"(index, unknown)",2


In [5]:
large_frequent_itemsets = frequent_itemsets[frequent_itemsets["length"] >= 3]
large_frequent_itemsets

,support,itemsets,length


In [6]:
import psycopg2

conn = psycopg2.connect(database="web_honeypot_vps", user = "postgres", password = "admin", host = "127.0.0.1", port = "5432")

print("Opened database successfully")

Opened database successfully


In [7]:
# create table
cur = conn.cursor()
cur.execute('''CREATE TABLE assoc_rules_attack_types (
            ID INT PRIMARY KEY     NOT NULL,
            SUPPORT           REAL    NOT NULL,
            ATTACK_TYPE            VARCHAR(255)     NOT NULL);''')

print("Table created successfully")

conn.commit()

Table created successfully


In [8]:
# insert data

cur = conn.cursor()

insert_query = """
    INSERT INTO assoc_rules_attack_types (ID, SUPPORT, ATTACK_TYPE)
    VALUES (%s, %s, %s)
"""

for idx, row in frequent_itemsets.iterrows():
    cur.execute(
        insert_query,
        (int(idx), float(row["support"]), str(list(row["itemsets"])))
    )

conn.commit()
print("Records created successfully")
conn.close()

Records created successfully


In [9]:
# create large data table
cur = conn.cursor()
cur.execute('''CREATE TABLE assoc_rules_attack_types_large (
            ID INT PRIMARY KEY     NOT NULL,
            SUPPORT           REAL    NOT NULL,
            ATTACK_TYPE            VARCHAR(255)     NOT NULL);''')

print("Table created successfully")

conn.commit()

InterfaceError: connection already closed

In [ ]:
# insert large data

cur = conn.cursor()

insert_query = """
    INSERT INTO assoc_rules_attack_types_large (ID, SUPPORT, ATTACK_TYPE)
    VALUES (%s, %s, %s)
"""

for idx, row in large_frequent_itemsets.iterrows():
    cur.execute(
        insert_query,
        (int(idx), float(row["support"]), str(list(row["itemsets"])))
    )

conn.commit()
print("Records created successfully")
conn.close()

Records created successfully
